In [3]:
import pandas as pd
import numpy as np

In [4]:
try:
	df = pd.read_csv("../../datasets/MMSD2.0/data/text_csv_clean_id/test.csv", encoding="utf-8")
except UnicodeDecodeError:
	df = pd.read_csv("../../datasets/MMSD2.0/data/text_csv_clean_id/test.csv", encoding="latin1")

df.head()

,image_id,text_en,text_id,label
0,862902619928506372,i am guessing # netflix no longer lets you gra...,"Menurutku, #Netflix sepertinya nggak lagi meng...",1
1,892551658487631873,it 's the insensitive strikeouts at suntrust p...,Itu adalah strikeout yang mengecewakan di SunT...,1
2,853143461360480256,"following the path of the river calder , so .....","menyusuri aliran Sungai Calder, jadi... suram....",1
3,918423568823840768,# westernsahara # authority has no lessons 2ge...,#sahara barat #pemerintah tidak perlu belajar ...,1
4,731617467718610944,hey <user> great sale !,"Hai <user>, obralnya keren banget!",1


In [5]:
#show some stats
print(df.describe())
print(df.info())

           image_id        label
count  2.373000e+03  2373.000000
mean   8.241311e+17     0.389802
std    5.205076e+16     0.487808
min    6.827168e+17     0.000000
25%    8.193237e+17     0.000000
50%    8.218690e+17     0.000000
75%    8.276195e+17     1.000000
max    9.465246e+17     1.000000
<class 'pandas.DataFrame'>
RangeIndex: 2373 entries, 0 to 2372
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   image_id  2373 non-null   int64
 1   text_en   2373 non-null   str  
 2   text_id   2373 non-null   str  
 3   label     2373 non-null   int64
dtypes: int64(2), str(2)
memory usage: 74.3 KB
None


# POC

In [6]:
import torch
from torch.utils.data import Dataset
from PIL import Image

In [ ]:
class SimpleSarcasmDataset(Dataset):
    def __init__(self, data, tokenizer, processor):
        self.data = data
        self.tokenizer = tokenizer
        self.processor = processor

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        text_inputs = self.tokenizer(item['text_id'], return_tensors="pt", padding='max_length', truncation=True, max_length=128)
        
        image = Image.open(f"../../datasets/MMSD2.0/data/dataset_image/{item['image_id']}.jpg").convert("RGB")
        image_inputs = self.processor(images=image, return_tensors="pt")
        
        return {
            'input_ids': text_inputs['input_ids'].squeeze(0),
            'pixel_values': image_inputs['pixel_values'].squeeze(0),
            'label': torch.tensor(item['label'], dtype=torch.float)
        }

In [ ]:
import torch.nn as nn

class MiniSarcasmModel(nn.Module):
    def __init__(self, text_model, vision_model):
        super().__init__()
        self.text_encoder = text_model
        self.vision_encoder = vision_model
        
        for param in self.text_encoder.parameters(): param.requires_grad = False
        for param in self.vision_encoder.parameters(): param.requires_grad = False

        self.fusion_layer = nn.Sequential(
            nn.Linear(768 + 768, 256),
            nn.ReLU(),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )

    def forward(self, input_ids, pixel_values):
        with torch.no_grad():
            t_feat = self.text_encoder(input_ids).pooler_output
            v_feat = self.vision_encoder(pixel_values).pooler_output
        
        combined = torch.cat((t_feat, v_feat), dim=1)
        return self.fusion_layer(combined)

In [11]:
from transformers import AutoTokenizer, AutoProcessor, AutoModel
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2")
processor = AutoProcessor.from_pretrained("google/vit-base-patch16-224")
text_model = AutoModel.from_pretrained("indobenchmark/indobert-base-p2")
vision_model = AutoModel.from_pretrained("google/vit-base-patch16-224")
dataset = SimpleSarcasmDataset(df.to_dict(orient='records'), tokenizer, processor)
model = MiniSarcasmModel(text_model, vision_model)
print(model)

/mnt/d/Student/8th semester (skripsi coyyyyyyy)/bukan-skripsi/notebooks/1st_exp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 198/198 [00:00<00:00, 223.45it/s]
ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MiniSarcasmModel(
  (text_encoder): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(50000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

In [12]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = nn.BCELoss()

In [ ]:
#train
epochs = 5
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    model.train()
    for item in dataset:
        input_ids = item['input_ids'].unsqueeze(0)
        pixel_values = item['pixel_values'].unsqueeze(0)
        label = item['label'].unsqueeze(0)
        
        output = model(input_ids, pixel_values)
        print(f"Output: {output.item():.4f}, Label: {label.item()}")
        loss = nn.BCELoss()(output, label)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
print("Done!")

Epoch 1
-------------------------------
Output: 0.5445, Label: 1.0


ValueError: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([1, 1])) is deprecated. Please ensure they have the same size.